# 3.2 Determinants and Elementary Row Operations

**Course:** Elementary Linear Algebra (Larson, 8e) 
**Chapter 3 — Determinants**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply, transpose, inverse
from linalg_core.determinant import det_cofactor, det_elimination
from linalg_core.elimination import swap_rows, scale_row, add_scaled_row
from linalg_core import EPSILON
import numpy as np
import random
import time

random.seed(42)
np.random.seed(42)


def random_matrix(n):
    """Generate a random n×n matrix with integer entries in [-9, 9]."""
    data = [float(random.randint(-9, 9)) for _ in range(n * n)]
    return Matrix(n, n, data)


def random_invertible(n):
    """Generate a random n×n invertible matrix (retry until det != 0)."""
    while True:
        A = random_matrix(n)
        if abs(det_elimination(A)) > EPSILON:
            return A


def to_numpy(M):
    """Convert our Matrix to a NumPy array for verification."""
    return np.array(M.to_lists())


print("Setup complete.")

---

## 2. Motivation

In Notebook 3.1 we computed determinants using **cofactor expansion**. That method
is elegant and gives us the Laplace expansion formula, but it is hopelessly slow
for large matrices.

Cofactor expansion is $O(n!)$. For a $20 \times 20$ matrix, that is roughly
$20! \approx 2.4 \times 10^{18}$ arithmetic operations. Even at a billion operations
per second, that would take **over 77 years**.

| Matrix size | Cofactor ops ($n!$) | Time @ $10^9$ ops/sec |
|:-----------:|:-------------------:|:---------------------:|
| $5 \times 5$ | 120 | instant |
| $10 \times 10$ | 3,628,800 | 0.004 s |
| $15 \times 15$ | $1.3 \times 10^{12}$ | 22 min |
| $20 \times 20$ | $2.4 \times 10^{18}$ | 77 years |

Can we do better? **Yes.** The three elementary row operations interact with
determinants in simple, predictable ways. By reducing a matrix to **upper
triangular form** via elimination, we can compute the determinant in $O(n^3)$ —
the same cost as Gaussian elimination itself.

This notebook derives and demonstrates each of these three rules, builds the
elimination-based determinant algorithm, and then proves several powerful
identities: $\det(AB) = \det(A)\det(B)$, $\det(A^T) = \det(A)$, and
$\det(A^{-1}) = 1/\det(A)$.

---

## 3. Build — Elementary Row Operations and the Determinant

We establish three theorems (Larson, Theorems 3.3–3.5) that describe
exactly how each elementary row operation affects the determinant.
These three rules are the engine behind efficient determinant computation.

### 3.1 Row Swap Negates the Determinant

**Theorem 3.3 (Larson).** If $B$ is obtained from $A$ by swapping two rows, then
$$\det(B) = -\det(A).$$

*Intuition:* A row swap reverses the "orientation" of the parallelepiped
defined by the row vectors. Orientation is a signed quantity, so it flips.

Let's verify with a concrete $3 \times 3$ example.

In [ ]:
A = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
])

det_before = det_cofactor(A)
print("A ="); print(A)
print(f"\ndet(A) = {det_before}")

B = A.copy()
swap_rows(B, 0, 2)
det_after = det_cofactor(B)
print("\nAfter swapping R1 <-> R3:")
print("B ="); print(B)
print(f"\ndet(B) = {det_after}")
print(f"\ndet(B) == -det(A)? {abs(det_after - (-det_before)) < EPSILON}")

C = B.copy()
swap_rows(C, 0, 1)
det_double = det_cofactor(C)
print("\nAfter a second swap (R1 <-> R2):")
print(f"det(C) = {det_double}")
print(f"Two swaps restore sign: det(C) == det(A)? {abs(det_double - det_before) < EPSILON}")

### 3.2 Scaling a Row Multiplies the Determinant

**Theorem 3.4 (Larson).** If $B$ is obtained from $A$ by multiplying one row
by a scalar $c$, then
$$\det(B) = c \cdot \det(A).$$

*Intuition:* Scaling one edge of the parallelepiped by $c$ scales the
enclosed volume by $|c|$ and preserves (or reverses, if $c < 0$) orientation.

**Corollary.** For an $n \times n$ matrix, $\det(cA) = c^n \det(A)$, because
every row gets scaled by $c$.

In [ ]:
A = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
])

det_A = det_cofactor(A)
c = 3.0
print(f"A ="); print(A)
print(f"\ndet(A) = {det_A}")

B = A.copy()
scale_row(B, 1, c)
det_B = det_cofactor(B)
print(f"\nAfter scaling R2 by {c}:")
print("B ="); print(B)
print(f"\ndet(B) = {det_B}")
print(f"c * det(A) = {c * det_A}")
print(f"det(B) == c * det(A)? {abs(det_B - c * det_A) < EPSILON}")

print("\n--- Corollary: det(cA) = c^n * det(A) ---")
from linalg_core.ops import scalar_mul
cA = scalar_mul(A, c)
det_cA = det_cofactor(cA)
n = A.rows
expected = (c ** n) * det_A
print(f"det({c}*A) = {det_cA}")
print(f"c^n * det(A) = {c}^{n} * {det_A} = {expected}")
print(f"Equal? {abs(det_cA - expected) < EPSILON}")

### 3.3 Row Addition Does Not Change the Determinant

**Theorem 3.5 (Larson).** If $B$ is obtained from $A$ by adding a scalar
multiple of one row to another row, then
$$\det(B) = \det(A).$$

*Intuition:* Adding a multiple of one edge to another edge of a
parallelepiped performs a "shear" — the shape tilts but the enclosed
volume does not change.

This is the workhorse operation in Gaussian elimination, and the fact
that it preserves the determinant is precisely what makes
elimination-based determinant computation possible.

In [ ]:
A = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
])

det_A = det_cofactor(A)
print("A ="); print(A)
print(f"\ndet(A) = {det_A}")

B = A.copy()
add_scaled_row(B, 2, 0, -7.0)
det_B = det_cofactor(B)
print("\nAfter R3 += (-7) * R1:")
print("B ="); print(B)
print(f"\ndet(B) = {det_B}")
print(f"det(B) == det(A)? {abs(det_B - det_A) < EPSILON}")

C = B.copy()
add_scaled_row(C, 1, 0, -4.0)
det_C = det_cofactor(C)
print("\nAfter another: R2 += (-4) * R1:")
print("C ="); print(C)
print(f"\ndet(C) = {det_C}")
print(f"Still equal? {abs(det_C - det_A) < EPSILON}")

### 3.4 The Elimination-Based Determinant

Now we combine the three rules into a complete algorithm.

**Algorithm:** To compute $\det(A)$:

1. Use Gaussian elimination (with partial pivoting) to reduce $A$ to
   upper triangular form $U$.
2. Track each row swap (each swap multiplies the determinant by $-1$).
3. Row additions don't change the determinant, so we ignore them.
4. The determinant of an upper triangular matrix is the product of its
   diagonal entries: $\det(U) = u_{11} u_{22} \cdots u_{nn}$.
5. Therefore: $\det(A) = (-1)^{\text{swaps}} \cdot u_{11} u_{22} \cdots u_{nn}$.

This is exactly what `det_elimination` implements. The cost is $O(n^3)$ —
the same as Gaussian elimination itself.

$$\det(A) = (-1)^s \prod_{i=1}^{n} u_{ii}$$

where $s$ is the number of row swaps and $u_{ii}$ are the diagonal entries
of the reduced upper triangular matrix.

In [ ]:
A = Matrix.from_lists([
    [ 2,  1,  3,  1],
    [ 4,  3,  8,  3],
    [ 6,  4, 12,  5],
    [ 2,  1,  5,  3]
])

print("A (4×4) ="); print(A)

det_cof = det_cofactor(A)
det_elim = det_elimination(A)

print(f"\ndet (cofactor expansion): {det_cof}")
print(f"det (elimination):        {det_elim}")
print(f"Match? {abs(det_cof - det_elim) < EPSILON}")

det_np = np.linalg.det(to_numpy(A))
print(f"\nNumPy verification: {det_np:.6f}")
print(f"Matches our result? {abs(det_elim - det_np) < 1e-6}")

print("\n--- Step-by-step view of elimination ---")
M = A.copy()
sign = 1
n = M.rows

for col in range(n):
    best = col
    for r in range(col + 1, n):
        if abs(M[r, col]) > abs(M[best, col]):
            best = r
    if best != col:
        swap_rows(M, col, best)
        sign *= -1
        print(f"Swap R{col+1} <-> R{best+1}  (sign -> {'+' if sign > 0 else '-'})")
    for r in range(col + 1, n):
        if abs(M[r, col]) > EPSILON:
            factor = M[r, col] / M[col, col]
            add_scaled_row(M, r, col, -factor)
    print(f"After column {col + 1} elimination:"); print(M); print()

diag_product = 1.0
for i in range(n):
    diag_product *= M[i, i]
result = sign * diag_product
print(f"Sign factor: {sign}")
print(f"Diagonal product: {diag_product}")
print(f"det(A) = ({sign}) * ({diag_product}) = {result}")

### 3.5 Speed Comparison — Cofactor vs. Elimination

The real payoff of elimination is speed. Cofactor expansion is $O(n!)$
while elimination is $O(n^3)$. Even at moderate sizes, the difference
is dramatic.

Let's time both methods on an $8 \times 8$ random matrix.

In [ ]:
A8 = random_matrix(8)
print("A (8×8) ="); print(A8)

t0 = time.perf_counter()
det_cof_8 = det_cofactor(A8)
t_cofactor = time.perf_counter() - t0

t0 = time.perf_counter()
det_elim_8 = det_elimination(A8)
t_elim = time.perf_counter() - t0

print(f"\nCofactor expansion:")
print(f"  det = {det_cof_8:.6f}")
print(f"  time = {t_cofactor:.4f} s")

print(f"\nElimination:")
print(f"  det = {det_elim_8:.6f}")
print(f"  time = {t_elim:.6f} s")

print(f"\nResults match? {abs(det_cof_8 - det_elim_8) < 1e-4}")
if t_elim > 0:
    print(f"Speedup: {t_cofactor / t_elim:.0f}x faster")
else:
    print("Elimination was too fast to measure!")

print(f"\n8! = {40320} operations (cofactor) vs ~8^3 = {8**3} operations (elimination)")
print("The gap grows factorially with n.")

### 3.6 Multiplicative Property: $\det(AB) = \det(A) \det(B)$

**Theorem 3.9 (Larson).** For any two $n \times n$ matrices $A$ and $B$,
$$\det(AB) = \det(A)\,\det(B).$$

This is one of the most important properties in all of linear algebra.
Among its many consequences:
- $\det(A^k) = [\det(A)]^k$
- If $A$ is invertible, $\det(A) \neq 0$ (and vice versa)
- The determinant is a **group homomorphism** from $GL(n, \mathbb{R})$ to $\mathbb{R}^*$

Let's verify with random invertible matrices.

In [ ]:
print("Verifying det(AB) = det(A) * det(B) with random 4×4 matrices:")
print("=" * 60)

random.seed(123)
all_pass = True

for trial in range(5):
    A = random_invertible(4)
    B = random_invertible(4)
    AB = multiply(A, B)

    det_A = det_elimination(A)
    det_B = det_elimination(B)
    det_AB = det_elimination(AB)
    product = det_A * det_B

    ok = abs(det_AB - product) < 1e-6
    all_pass = all_pass and ok
    print(f"  Trial {trial+1}: det(A)={det_A:10.4f}  det(B)={det_B:10.4f}  "
          f"det(AB)={det_AB:12.4f}  det(A)*det(B)={product:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\nAlso works for singular matrices:")
S = Matrix.from_lists([[1, 2, 3], [4, 5, 6], [5, 7, 9]])
R = random_invertible(3)
SR = multiply(S, R)
det_S = det_elimination(S)
det_R = det_elimination(R)
det_SR = det_elimination(SR)
print(f"  det(S) = {det_S} (singular)")
print(f"  det(R) = {det_R:.4f}")
print(f"  det(SR) = {det_SR}")
print(f"  det(S)*det(R) = {det_S * det_R}")
print(f"  Match? {abs(det_SR - det_S * det_R) < EPSILON}")

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

### 3.7 Transpose Property: $\det(A^T) = \det(A)$

**Theorem 3.6 (Larson).** For any square matrix $A$,
$$\det(A^T) = \det(A).$$

This means **every row theorem has a column counterpart**. Swapping two
*columns* also negates the determinant; scaling a *column* by $c$ multiplies
the determinant by $c$; adding a scalar multiple of one *column* to another
leaves the determinant unchanged.

The transpose property also explains why cofactor expansion works along
any row *or column*.

In [ ]:
print("Verifying det(A^T) = det(A) with random matrices:")
print("=" * 60)

random.seed(456)
all_pass = True

for trial in range(5):
    n = random.choice([3, 4, 5])
    A = random_matrix(n)
    At = transpose(A)

    det_A = det_elimination(A)
    det_At = det_elimination(At)

    ok = abs(det_A - det_At) < 1e-6
    all_pass = all_pass and ok
    print(f"  Trial {trial+1} ({n}×{n}): det(A)={det_A:12.4f}  det(A^T)={det_At:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\nNon-square example (det is only defined for square):")
A_sq = Matrix.from_lists([[1, 2, 3], [0, 4, 5], [0, 0, 6]])
At_sq = transpose(A_sq)
print(f"  Upper triangular A: det = {det_elimination(A_sq)}")
print(f"  Lower triangular A^T: det = {det_elimination(At_sq)}")
print(f"  Equal? {abs(det_elimination(A_sq) - det_elimination(At_sq)) < EPSILON}")

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

### 3.8 Inverse Property: $\det(A^{-1}) = \dfrac{1}{\det(A)}$

**Theorem (Larson, Sec. 3.2).** If $A$ is invertible, then
$$\det(A^{-1}) = \frac{1}{\det(A)}.$$

*Proof (one line):* Since $A A^{-1} = I$, by the multiplicative property:
$$\det(A)\,\det(A^{-1}) = \det(I) = 1$$
so $\det(A^{-1}) = 1/\det(A)$. $\square$

This also confirms: $A$ is invertible $\iff$ $\det(A) \neq 0$.

In [ ]:
print("Verifying det(A⁻¹) = 1/det(A) with random invertible matrices:")
print("=" * 60)

random.seed(789)
all_pass = True

for trial in range(5):
    n = random.choice([3, 4, 5])
    A = random_invertible(n)
    A_inv = inverse(A)

    det_A = det_elimination(A)
    det_Ainv = det_elimination(A_inv)
    reciprocal = 1.0 / det_A

    ok = abs(det_Ainv - reciprocal) < 1e-4
    all_pass = all_pass and ok
    print(f"  Trial {trial+1} ({n}×{n}): det(A)={det_A:10.4f}  "
          f"det(A⁻¹)={det_Ainv:12.6f}  1/det(A)={reciprocal:12.6f}  {'PASS' if ok else 'FAIL'}")

print(f"\nSanity check: det(A) * det(A⁻¹) should equal 1:")
A = random_invertible(4)
A_inv = inverse(A)
product = det_elimination(A) * det_elimination(A_inv)
print(f"  det(A) * det(A⁻¹) = {product:.10f}")
print(f"  ≈ 1? {abs(product - 1.0) < 1e-6}")

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 4. Verify — Cross-Check Everything

We run systematic checks across all the properties demonstrated above.
Every result from our from-scratch library is verified against both
`det_cofactor` and `np.linalg.det`.

In [ ]:
print("=" * 60)
print("VERIFICATION 1: det_elimination vs det_cofactor")
print("=" * 60)

random.seed(1000)
all_pass = True

for trial in range(10):
    n = random.choice([2, 3, 4, 5])
    A = random_matrix(n)
    d_cof = det_cofactor(A)
    d_elim = det_elimination(A)
    ok = abs(d_cof - d_elim) < 1e-6
    all_pass = all_pass and ok
    print(f"  Trial {trial+1:2d} ({n}×{n}): cofactor={d_cof:12.4f}  elim={d_elim:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\n  Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 2: det_elimination vs np.linalg.det")
print("=" * 60)

random.seed(2000)
all_pass = True

for trial in range(10):
    n = random.choice([3, 4, 5, 6, 7])
    A = random_matrix(n)
    d_elim = det_elimination(A)
    d_np = np.linalg.det(to_numpy(A))
    ok = abs(d_elim - d_np) < 1e-4
    all_pass = all_pass and ok
    print(f"  Trial {trial+1:2d} ({n}×{n}): ours={d_elim:12.4f}  numpy={d_np:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\n  Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 3: det(AB) = det(A)*det(B)")
print("=" * 60)

random.seed(3000)
all_pass = True

for trial in range(10):
    n = random.choice([3, 4, 5])
    A = random_matrix(n)
    B = random_matrix(n)
    AB = multiply(A, B)
    lhs = det_elimination(AB)
    rhs = det_elimination(A) * det_elimination(B)
    ok = abs(lhs - rhs) < 1e-4
    all_pass = all_pass and ok
    print(f"  Trial {trial+1:2d} ({n}×{n}): det(AB)={lhs:12.4f}  det(A)*det(B)={rhs:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\n  Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 4: det(A^T) = det(A)")
print("=" * 60)

random.seed(4000)
all_pass = True

for trial in range(10):
    n = random.choice([3, 4, 5, 6])
    A = random_matrix(n)
    At = transpose(A)
    d_A = det_elimination(A)
    d_At = det_elimination(At)
    ok = abs(d_A - d_At) < 1e-6
    all_pass = all_pass and ok
    print(f"  Trial {trial+1:2d} ({n}×{n}): det(A)={d_A:12.4f}  det(A^T)={d_At:12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\n  Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 5: All Three Row-Operation Rules")
print("=" * 60)

random.seed(5000)
all_pass = True

for trial in range(5):
    A = random_invertible(4)
    det_A = det_elimination(A)

    B_swap = A.copy()
    swap_rows(B_swap, 0, 2)
    ok_swap = abs(det_cofactor(B_swap) - (-det_A)) < 1e-6

    c = 5.0
    B_scale = A.copy()
    scale_row(B_scale, 1, c)
    ok_scale = abs(det_cofactor(B_scale) - c * det_A) < 1e-6

    B_add = A.copy()
    add_scaled_row(B_add, 2, 0, -3.0)
    ok_add = abs(det_cofactor(B_add) - det_A) < 1e-6

    ok = ok_swap and ok_scale and ok_add
    all_pass = all_pass and ok
    print(f"  Trial {trial+1}: swap={'PASS' if ok_swap else 'FAIL'}  "
          f"scale={'PASS' if ok_scale else 'FAIL'}  "
          f"add={'PASS' if ok_add else 'FAIL'}")

print(f"\n  Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy) — Triangular Shortcut

The determinant of a **triangular matrix** (upper or lower) is simply the
product of its diagonal entries.

$$\det(T) = t_{11}\, t_{22}\, \cdots\, t_{nn}$$

Verify this for the following matrices by:
1. Computing the diagonal product by hand (in a loop).
2. Comparing with `det_elimination`.

$$U = \begin{bmatrix} 3 & 1 & 4 \\ 0 & 2 & 7 \\ 0 & 0 & 5 \end{bmatrix}, \quad
  L = \begin{bmatrix} -2 & 0 & 0 & 0 \\ 3 & 4 & 0 & 0 \\ 1 & -1 & 6 & 0 \\ 5 & 2 & 8 & -1 \end{bmatrix}$$

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium) — Which Matrices Are Invertible?

A matrix is invertible if and only if its determinant is nonzero.

For each matrix below, compute $\det(A)$ using `det_elimination` and
determine whether it is invertible. For the invertible ones, also verify
that $\det(A^{-1}) = 1/\det(A)$.

$$A_1 = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{bmatrix}, \quad
  A_2 = \begin{bmatrix} 2 & 0 & -1 \\ 0 & 3 & 0 \\ 4 & 0 & 1 \end{bmatrix}, \quad
  A_3 = \begin{bmatrix} 1 & 1 & 1 & 1 \\ 1 & 2 & 3 & 4 \\ 1 & 3 & 6 & 10 \\ 1 & 4 & 10 & 20 \end{bmatrix}$$

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) — Benchmark and Plot

Benchmark `det_cofactor` and `det_elimination` for matrix sizes
$n = 2, 3, 4, \ldots, 12$. For each size:
1. Generate a random $n \times n$ matrix.
2. Time both methods (use `time.perf_counter()`).
3. Store the times.

Then plot both curves on the same axes using `matplotlib`.
Use a **log scale** for the $y$-axis to see the exponential vs. polynomial
growth clearly.

*Hint:* You may want to skip `det_cofactor` for $n > 10$ if it takes
too long. Use a timeout or just record `None` for those sizes.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Elementary Row Operation | Effect on $\det$ | Cost to track |
|:---|:---|:---|
| Swap two rows | $\det \to -\det$ | Flip a sign bit |
| Scale row $i$ by $c$ | $\det \to c \cdot \det$ | Multiply by $c$ |
| Add $c \cdot R_j$ to $R_i$ | $\det$ unchanged | Nothing |

**Key results:**

| Property | Statement |
|:---|:---|
| Triangular det | $\det(T) = \prod_{i} t_{ii}$ |
| Multiplicative | $\det(AB) = \det(A)\,\det(B)$ |
| Transpose | $\det(A^T) = \det(A)$ |
| Inverse | $\det(A^{-1}) = 1/\det(A)$ |
| Powers | $\det(A^k) = [\det(A)]^k$ |
| Scalar | $\det(cA) = c^n \det(A)$ |

**Complexity comparison:**

| Method | Cost | Practical limit |
|:---|:---|:---|
| Cofactor expansion | $O(n!)$ | $n \leq 10$ |
| Elimination | $O(n^3)$ | $n \leq 10{,}000+$ |

**Takeaway:** The three elementary row operations give us a complete toolkit
for computing determinants efficiently. Elimination reduces any matrix to
upper triangular form in $O(n^3)$ time, and the only bookkeeping needed is
counting row swaps. This transforms the determinant from a theoretical
curiosity into a practical computational tool.